# Healthcare Data Governance & Quality Analytics
**Privacy-Aware Patient Data Validation, Cleaning & Quality Monitoring** - EdVyro Data Analytics Program, Task 2.

This notebook runs the same reusable modules as the dashboard (`data_processor.py`, `validation_rules.py`, `data_dictionary.py`).
Every number below is computed when you run the cell - nothing is typed in by hand.

> Clinical thresholds are data-quality screens, not diagnostic criteria. No medical conclusions are drawn.

## 1. Setup & dataset source
The notebook tries the **official EdVyro URL** first. If it is unreachable it falls back to the local copy in `dataset/` and states which one was used.

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import config, data_processor as dp, validation_rules as vr
from data_dictionary import build_data_dictionary, privacy_assessment
sns.set_theme(style="whitegrid", palette=["#1D6FDC", "#0E9AA7", "#5FD3BC", "#2FB36D"])
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

## 2. Data loading

In [ ]:
raw, info = dp.load_with_fallback()
print("Source type :", info["source_type"])
print("Location    :", info["source_label"])
print("File size   :", info["file_size_bytes"], "bytes | encoding:", info["encoding"])
if info["note"]: print("NOTE        :", info["note"])
raw_snapshot = raw.copy(deep=True)   # proves later that the original is never modified

## 3. Dataset overview

In [ ]:
print("Shape:", raw.shape)
display(raw.head(10))
display(raw.dtypes.rename("dtype").to_frame())
display(raw.describe(include="all").T)

## 4. Privacy assessment
Based on column names and value patterns. The dataset is **never** declared fully de-identified unless documentation supports it.

In [ ]:
priv = privacy_assessment(raw)
print("Status        :", priv["status"])
print("Why           :", priv["explanation"])
print("Patient IDs   :", priv["pseudonymous_identifiers"])
print("Quasi-ids     :", priv["quasi_identifiers"])
print("Sensitive cols:", priv["sensitive_columns"])
print("k-anonymity   :", priv["k_anonymity"])
print("\nRemaining risks:"); [print(" -", r) for r in priv["risks"]]
print("\nSafeguards:");      [print(" -", r) for r in priv["safeguards"]]

## 5. Run the full pipeline
One call validates, cleans, scores and builds the dictionary and exception log. The steps below then inspect each part.

In [ ]:
res = dp.run_pipeline(raw, info, enable_anomaly=True)
assert raw.equals(raw_snapshot), "original dataset must be unchanged"
print("Original dataset unchanged:", raw.equals(raw_snapshot))
display(pd.DataFrame(res.steps)[["step", "status", "affected"]])

## 6. Data dictionary

In [ ]:
display(res.dictionary)

## 7. Missing-value analysis

In [ ]:
mask = vr.missing_mask(raw)
miss = pd.DataFrame({"missing": mask.sum(), "missing_pct": (mask.mean() * 100).round(2)})
display(miss)
fig, ax = plt.subplots(figsize=(8, 3))
sns.heatmap(mask.T.astype(int), cbar=False, cmap=["#DDF5E8", "#E0525A"], ax=ax)
ax.set_title("Missing-value heatmap (red = missing)"); ax.set_xlabel("row"); plt.show()
print("Total missing cells:", int(mask.values.sum()))

## 8. Duplicate analysis
Duplicates are flagged, never auto-deleted - a repeated patient ID may be a legitimate repeat encounter.

In [ ]:
id_col = res.stats["id_col"]
print("ID column:", id_col)
print("Rows sharing a patient ID :", int(raw[id_col].duplicated(keep=False).sum()) if id_col else "n/a")
print("Exact duplicate rows      :", int(raw.duplicated(keep=False).sum()))
display(res.exceptions[res.exceptions["issue_category"].isin(["Duplicate patient ID", "Duplicate record"])])

## 9. Data type validation

In [ ]:
display(res.dictionary[["Column Name", "Data Type", "Cleaned Column Name", "Cleaned Data Type"]])
display(res.rules[res.rules["rule"].isin(["Column types match expectation", "Numeric conversion valid", "Blood pressure format"])])

## 10. Numeric validation (negatives, conversion, outliers)

In [ ]:
num = [c for c in res.stats["numeric_cols"]]
display(raw[num].apply(pd.to_numeric, errors="coerce").describe().T)
display(res.rules[res.rules["rule"].isin(["Numeric conversion valid", "No negative values where prohibited", "IQR outlier screen"])])
display(res.exceptions[res.exceptions["issue_category"] == "Statistical outlier"])

## 11. Clinical value validation (data-quality screens only)
Thresholds come from `config.CLINICAL_RULES`. Rules for columns that do not exist (heart rate, BMI, glucose, temperature) are skipped automatically.

In [ ]:
display(pd.DataFrame(config.CLINICAL_RULES).T[["label", "min", "max", "unit", "severity"]])
display(res.rules[res.rules["category"].isin(["Out-of-range value", "Invalid data type"]) & res.rules["dimension"].eq("Validity")])

## 12. Categorical validation

In [ ]:
for c in res.stats["categorical_cols"]:
    print(c, "->", raw[c].value_counts().to_dict())
display(res.rules[res.rules["dimension"].eq("Consistency") | res.rules["category"].eq("Invalid category")])

## 13. Data cleaning
Safe steps only: normalise names, trim whitespace, unify case-only category variants, convert types when *every* value converts, derive systolic/diastolic from the BP text. Nothing is deleted or imputed; clinical values are verified unchanged.

In [ ]:
for s in res.steps:
    print(f"{s['step']:<62} {s['status']:<18} affected={s['affected']}")
print("\nCells modified by cleaning:", len(res.changes))
display(res.changes if len(res.changes) else pd.DataFrame({"result": ["cleaning modified 0 cells"]}))
display(res.cleaned.head())

## 14. Exception log

In [ ]:
display(res.exceptions)
display(res.exceptions["severity"].value_counts().rename("count").to_frame())

## 15. Data quality scoring

In [ ]:
print(config.SCORE_FORMULA_TEXT, "\n")
sb = res.scores_before
display(pd.Series(sb["dimensions"], name="score").to_frame())
print("Overall:", sb["overall"], "| exceptions:", sb["total_exceptions"], "| issues above Informational:", sb["issues_detected"],
      "| records needing review:", sb["records_requiring_review"], "| validated records:", sb["validated_records"])

## 16. Before / after comparison

In [ ]:
cmp = pd.DataFrame({"original": res.scores_before["dimensions"], "cleaned": res.scores_after["dimensions"]})
cmp.loc["OVERALL"] = [res.scores_before["overall"], res.scores_after["overall"]]
display(cmp)
print("Shape before:", raw.shape, "| shape after (incl. derived + flag columns):", res.cleaned.shape)

## 17. Visualisations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
cmp.drop("OVERALL").plot.bar(ax=axes[0, 0], ylim=(0, 105), title="Quality dimensions"); axes[0, 0].tick_params(axis="x", rotation=30)
miss["missing"].plot.bar(ax=axes[0, 1], title="Missing values by column", color="#0E9AA7")
sev = res.exceptions["severity"].value_counts().reindex(config.SEVERITY_ORDER).fillna(0)
sev.plot.bar(ax=axes[0, 2], title="Exceptions by severity", color="#1D6FDC")
res.dictionary["Data Classification"].value_counts().plot.bar(ax=axes[1, 0], title="Data classification", color="#2FB36D")
res.dictionary["Data Type"].value_counts().plot.pie(ax=axes[1, 1], title="Data types", ylabel="")
col = num[0]; sns.histplot(pd.to_numeric(raw[col], errors="coerce").dropna(), bins=10, kde=False, ax=axes[1, 2]).set_title(f"Distribution: {col}")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 3.5))
sns.boxplot(data=raw[num].apply(pd.to_numeric, errors="coerce").apply(lambda s: (s - s.mean()) / s.std()), ax=ax, orient="h")
ax.set_title("Outlier view (z-scored so columns share one axis)"); plt.show()

### Optional: Isolation Forest anomaly scores
Statistical only (not medical). No accuracy metric is reported because there are no ground-truth labels. Identifiers are excluded; no value is changed.

In [ ]:
print(res.anomaly_info["status"], "| features:", res.anomaly_info.get("features"))
if res.anomalies is not None:
    display(res.anomalies.sort_values("anomaly_score"))
    for note in res.anomaly_info["limitations"]: print(" -", note)

## 18. Findings

In [ ]:
sb = res.scores_before
findings = [
    f"Dataset: {raw.shape[0]} rows x {raw.shape[1]} columns (source: {info['source_type']}).",
    f"Missing cells: {sb['missing_values']}; rows sharing a patient ID: {sb['duplicate_id_rows']}; duplicate-record flags: {sb['duplicate_record_rows']}.",
    f"Invalid cells (type/range/category): {sb['invalid_values']}; records needing review: {sb['records_requiring_review']}.",
    f"Overall quality score: {sb['overall']} / 100 ({sb['issues_detected']} issues above Informational; {sb['total_exceptions']} total exceptions).",
    f"Cleaning modified {len(res.changes)} cell(s); clinical values verified unchanged: {next(s['after'] for s in res.steps if s['step'].startswith('10.'))}.",
    f"Privacy: {priv['status']}.",
]
display(Markdown("\n".join("- " + f for f in findings)))

## 19. Recommendations
1. **Treat the privacy findings as the main governance issue**, not the quality score: exact ages plus gender can single out individuals in a small sample - generalise to age bands before sharing.
2. Keep patient IDs pseudonymous, mask them in displays, and store any re-identification key separately.
3. Obtain an official data dictionary (units, allowed values, collection method) so descriptions and units are documented rather than assumed.
4. Re-run this notebook on any new extract; thresholds in `config.py` can be tuned per clinical-data owner.
5. Never impute or auto-correct clinical values - route flagged records to the data owner.

## 20. Conclusion
The pipeline loads the dataset, classifies fields by privacy sensitivity, validates structure, completeness, uniqueness, ranges and categories,
applies only safe documented cleaning, logs every finding and reports a transparent quality score. The scores reported above describe **this dataset only**;
a high score on a very small sample is not evidence that the process would find no problems on larger or messier data (see `tests/` for demonstrations on deliberately defective frames).

*Educational data-quality analysis only. Not for diagnosis or clinical decision-making.*